# Register an Azure ML Environment

Generate a Conda file in the notebook, define a versioned environment from it and a base image, register it, and retrieve it for verification.

**Source:** Adapted from [Azure/azureml-examples environment.ipynb](https://github.com/Azure/azureml-examples/blob/7dbe9a3ddfc4a920a9de82eaa4af7eaf118841d8/sdk/python/assets/environment/environment.ipynb), MIT License.

In [7]:
from pathlib import Path
import os
import textwrap
import yaml

from azure.ai.ml import MLClient
from azure.ai.ml.entities import Environment
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "outputs").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None)
ml_client = MLClient(credential, os.environ["AZURE_SUBSCRIPTION_ID"], os.environ["AZURE_RESOURCE_GROUP"], os.environ["AZUREML_WORKSPACE_NAME"])
ENVIRONMENT_NAME = os.environ["WORKSHOP_ENVIRONMENT_NAME"]
REGISTER = os.getenv("REGISTER_FOUNDATION_ENVIRONMENT", "false").lower() in {"1", "true", "yes"}

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [8]:
import requests

BASE_IMAGE = "mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest"
manifest_url = "https://mcr.microsoft.com/v2/azureml/openmpi4.1.0-ubuntu20.04/manifests/latest"
manifest_response = requests.get(
    manifest_url,
    headers={"Accept": "application/vnd.docker.distribution.manifest.v2+json"},
    timeout=30,
 )
if manifest_response.status_code != 200:
    raise RuntimeError(
        f"Base image manifest is unavailable: {BASE_IMAGE} "
        f"(HTTP {manifest_response.status_code})"
    )
print(f"Verified base image manifest: {BASE_IMAGE}")

generated_dir = WORKSHOP_ROOT / "outputs/generated/foundations/environment"
generated_dir.mkdir(parents=True, exist_ok=True)
conda_file = generated_dir / "conda.yaml"
conda_content = textwrap.dedent("""
name: azureml-workshop-foundations
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pip:
      - azureml-inference-server-http==1.4.1
      - pandas==2.2.3
""").lstrip()
yaml.safe_load(conda_content)
conda_file.write_text(conda_content, encoding="utf-8")
print(f"Generated Conda file: {conda_file}")
print(conda_content)

environment_definition = Environment(
    name=ENVIRONMENT_NAME,
    image=BASE_IMAGE,
    conda_file=str(conda_file),
    description="Small workshop environment for command and endpoint demonstrations",
    tags={"workshop": "azureml-h2o", "purpose": "foundations"},
)

if REGISTER:
    registered_environment = ml_client.environments.create_or_update(environment_definition)
    verified_environment = ml_client.environments.get(
        ENVIRONMENT_NAME,
        version=registered_environment.version,
    )
    assert verified_environment.name == ENVIRONMENT_NAME
    assert verified_environment.version == registered_environment.version
    assert verified_environment.image == BASE_IMAGE
    print(f"Registered environment asset: {verified_environment.name}:{verified_environment.version}")
    print("Azure ML requested environment image materialization asynchronously.")
    print("Asset retrieval confirms registration, not image-build completion; monitor the build status and logs in Azure ML Studio.")
else:
    print(f"Prepared environment definition: {ENVIRONMENT_NAME} (version assigned on registration)")
    print("Registration disabled. Set REGISTER_FOUNDATION_ENVIRONMENT=true in workshop/.env.")

Verified base image manifest: mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest
Generated Conda file: /mnt/batch/tasks/shared/LS_root/mounts/clusters/aml-instance-dev-cc01/code/MLOPs-AzureML-backup-a7a4bc8-20260915/workshop/outputs/generated/foundations/environment/conda.yaml
name: azureml-workshop-foundations
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pip:
      - azureml-inference-server-http==1.4.1
      - pandas==2.2.3

Registered environment asset: workshop-taxi-environment:2
Azure ML requested environment image materialization asynchronously.
Asset retrieval confirms registration, not image-build completion; monitor the build status and logs in Azure ML Studio.


## Expected Result

The notebook-generated Conda specification defines a named, immutable Azure ML environment. Azure ML assigns its version during registration.

Next: `04_register_model.ipynb`.